In [1]:
import pathlib
from datetime import datetime

import numpy as np
import xarray as xr

from tobac_flow.analysis import get_label_stats, weighted_statistics_on_labels
from tobac_flow.dataset import calculate_label_properties
from tobac_flow.linking import process_file
from tobac_flow.utils.datetime_utils import get_dates_from_filename
from tobac_flow.utils.xarray_utils import add_compression_encoding, add_dataarray_to_ds

from tobac_flow.dataset import (
    add_label_coords,
    add_step_labels,
    add_label_coords,
    flag_edge_labels,
    flag_nan_adjacent_labels,
    calculate_label_properties,
    link_cores_and_anvils,
    link_step_labels
)
from tobac_flow.analysis import (
    get_label_stats,
    weighted_statistics_on_labels,
)
from tobac_flow.utils import (
    add_dataarray_to_ds,
    create_dataarray
)
from tobac_flow.utils.label_utils import labeled_comprehension

In [2]:
dataset = xr.open_dataset("../scripts/synsat_tracking_zoom9_2021.nc")

In [3]:
dataset.load()

<xarray.Dataset> Size: 18GB
Dimensions:             (t: 660, lat: 1500, lon: 1500)
Coordinates:
    crs                 int64 8B 0
  * t                   (t) datetime64[ns] 5kB 2021-07-01T00:15:00 ... 2021-0...
    cell                (lat, lon) int64 18MB 2916805 2916805 ... 228922 228922
  * lat                 (lat) float64 12kB -74.95 -74.85 -74.75 ... 74.85 74.95
  * lon                 (lon) float64 12kB 285.1 285.1 285.2 ... 74.85 74.95
Data variables:
    core_labels         (t, lat, lon) int32 6GB 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0
    thick_anvil_labels  (t, lat, lon) int32 6GB 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0
    thin_anvil_labels   (t, lat, lon) int32 6GB 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0

In [4]:
dataset = dataset.rename(
    core_labels="core_label",
    thick_anvil_labels="thick_anvil_label",
    thin_anvil_labels="thin_anvil_label",
)

In [5]:
ind = list(range(660))
ind.remove(471)

In [6]:
dataset = dataset.isel(t=ind)

In [7]:
dataset

<xarray.Dataset> Size: 18GB
Dimensions:            (t: 659, lat: 1500, lon: 1500)
Coordinates:
    crs                int64 8B 0
  * t                  (t) datetime64[ns] 5kB 2021-07-01T00:15:00 ... 2021-07-08
    cell               (lat, lon) int64 18MB 2916805 2916805 ... 228922 228922
  * lat                (lat) float64 12kB -74.95 -74.85 -74.75 ... 74.85 74.95
  * lon                (lon) float64 12kB 285.1 285.1 285.2 ... 74.85 74.95
Data variables:
    core_label         (t, lat, lon) int32 6GB 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0
    thick_anvil_label  (t, lat, lon) int32 6GB 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0
    thin_anvil_label   (t, lat, lon) int32 6GB 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0

In [8]:
dataset = add_label_coords(dataset)

In [9]:
dataset

<xarray.Dataset> Size: 18GB
Dimensions:            (t: 659, lat: 1500, lon: 1500, core: 6997, anvil: 10021)
Coordinates:
    crs                int64 8B 0
  * t                  (t) datetime64[ns] 5kB 2021-07-01T00:15:00 ... 2021-07-08
    cell               (lat, lon) int64 18MB 2916805 2916805 ... 228922 228922
  * lat                (lat) float64 12kB -74.95 -74.85 -74.75 ... 74.85 74.95
  * lon                (lon) float64 12kB 285.1 285.1 285.2 ... 74.85 74.95
  * core               (core) int32 28kB 1 2 3 4 5 ... 6993 6994 6995 6996 6997
  * anvil              (anvil) int32 40kB 1 2 3 4 5 ... 10018 10019 10020 10021
Data variables:
    core_label         (t, lat, lon) int32 6GB 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0
    thick_anvil_label  (t, lat, lon) int32 6GB 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0
    thin_anvil_label   (t, lat, lon) int32 6GB 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0

In [10]:
link_cores_and_anvils(dataset)

In [11]:
dataset

<xarray.Dataset> Size: 18GB
Dimensions:            (t: 659, lat: 1500, lon: 1500, core: 6997, anvil: 10021)
Coordinates:
    crs                int64 8B 0
  * t                  (t) datetime64[ns] 5kB 2021-07-01T00:15:00 ... 2021-07-08
    cell               (lat, lon) int64 18MB 2916805 2916805 ... 228922 228922
  * lat                (lat) float64 12kB -74.95 -74.85 -74.75 ... 74.85 74.95
  * lon                (lon) float64 12kB 285.1 285.1 285.2 ... 74.85 74.95
  * core               (core) int32 28kB 1 2 3 4 5 ... 6993 6994 6995 6996 6997
  * anvil              (anvil) int32 40kB 1 2 3 4 5 ... 10018 10019 10020 10021
Data variables:
    core_label         (t, lat, lon) int32 6GB 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0
    thick_anvil_label  (t, lat, lon) int32 6GB 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0
    thin_anvil_label   (t, lat, lon) int32 6GB 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0
    core_anvil_index   (core) int32 28kB 65 152 117 173 144 ... 9795 0 0 0 9564
    anvil_core_count   (anvil) int32 40kB 0 0 7 0 0 0 0 0 0 ... 1 1 0 0 0 0 0 0

In [12]:
add_step_labels(dataset)

dataset = add_label_coords(dataset)

In [13]:
dataset

<xarray.Dataset> Size: 36GB
Dimensions:                 (t: 659, lat: 1500, lon: 1500, core: 6997,
                             anvil: 10021, y: 1500, x: 1500, core_step: 29853,
                             thick_anvil_step: 262469, thin_anvil_step: 310095)
Coordinates:
    crs                     int64 8B 0
  * t                       (t) datetime64[ns] 5kB 2021-07-01T00:15:00 ... 20...
    cell                    (lat, lon) int64 18MB 2916805 2916805 ... 228922
  * lat                     (lat) float64 12kB -74.95 -74.85 ... 74.85 74.95
  * lon                     (lon) float64 12kB 285.1 285.1 285.2 ... 74.85 74.95
  * core                    (core) int32 28kB 1 2 3 4 5 ... 6994 6995 6996 6997
  * anvil                   (anvil) int32 40kB 1 2 3 4 ... 10019 10020 10021
  * core_step               (core_step) int32 119kB 1 2 3 ... 29851 29852 29853
  * thick_anvil_step        (thick_anvil_step) int32 1MB 1 2 3 ... 262468 262469
  * thin_anvil_step         (thin_anvil_step) int32 1MB 1 2 3 ... 310094 310095
Dimensions without coordinates: y, x
Data variables:
    core_label              (t, lat, lon) int32 6GB 0 0 0 0 0 0 ... 0 0 0 0 0 0
    thick_anvil_label       (t, lat, lon) int32 6GB 0 0 0 0 0 0 ... 0 0 0 0 0 0
    thin_anvil_label        (t, lat, lon) int32 6GB 0 0 0 0 0 0 ... 0 0 0 0 0 0
    core_anvil_index        (core) int32 28kB 65 152 117 173 144 ... 0 0 0 9564
    anvil_core_count        (anvil) int32 40kB 0 0 7 0 0 0 0 0 ... 1 0 0 0 0 0 0
    core_step_label         (t, y, x) int32 6GB 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0
    thick_anvil_step_label  (t, y, x) int32 6GB 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0
    thin_anvil_step_label   (t, y, x) int32 6GB 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0

In [31]:
link_step_labels(dataset)

In [32]:
dataset

<xarray.Dataset> Size: 36GB
Dimensions:                       (t: 659, lat: 1500, lon: 1500, core: 6997,
                                   anvil: 10021, y: 1500, x: 1500,
                                   core_step: 29853, thick_anvil_step: 262469,
                                   thin_anvil_step: 310095)
Coordinates:
    crs                           int64 8B 0
  * t                             (t) datetime64[ns] 5kB 2021-07-01T00:15:00 ...
    cell                          (lat, lon) int64 18MB 2916805 ... 228922
  * lat                           (lat) float64 12kB -74.95 -74.85 ... 74.95
  * lon                           (lon) float64 12kB 285.1 285.1 ... 74.85 74.95
  * core                          (core) int32 28kB 1 2 3 4 ... 6995 6996 6997
  * anvil                         (anvil) int32 40kB 1 2 3 ... 10019 10020 10021
  * core_step                     (core_step) int32 119kB 1 2 3 ... 29852 29853
  * thick_anvil_step              (thick_anvil_step) int32 1MB 1 2 ... 262469
  * thin_anvil_step               (thin_anvil_step) int32 1MB 1 2 ... 310095
Dimensions without coordinates: y, x
Data variables: (12/44)
    core_label                    (t, lat, lon) int32 6GB 0 0 0 0 0 ... 0 0 0 0
    thick_anvil_label             (t, lat, lon) int32 6GB 0 0 0 0 0 ... 0 0 0 0
    thin_anvil_label              (t, lat, lon) int32 6GB 0 0 0 0 0 ... 0 0 0 0
    core_anvil_index              (core) int32 28kB 65 152 117 173 ... 0 0 9564
    anvil_core_count              (anvil) int32 40kB 0 0 7 0 0 0 ... 0 0 0 0 0 0
    core_step_label               (t, y, x) int32 6GB 0 0 0 0 0 0 ... 0 0 0 0 0
    ...                            ...
    thin_anvil_step_pixel_count   (thin_anvil_step) int32 1MB 6096 1704 ... 97
    thin_anvil_step_area          (thin_anvil_step) float32 1MB 3.702e+05 ......
    thin_anvil_start_t            (anvil) datetime64[ns] 80kB 2021-07-01T00:1...
    thin_anvil_end_t              (anvil) datetime64[ns] 80kB 2021-07-02T09:1...
    thin_anvil_lifetime           (anvil) timedelta64[ns] 80kB 1 days 09:00:0...
    thin_anvil_step_t             (thin_anvil_step) datetime64[ns] 2MB 2021-0...

In [33]:
flag_edge_labels(dataset)

In [34]:
dataset

<xarray.Dataset> Size: 36GB
Dimensions:                       (t: 659, lat: 1500, lon: 1500, core: 6997,
                                   anvil: 10021, y: 1500, x: 1500,
                                   core_step: 29853, thick_anvil_step: 262469,
                                   thin_anvil_step: 310095)
Coordinates:
    crs                           int64 8B 0
  * t                             (t) datetime64[ns] 5kB 2021-07-01T00:15:00 ...
    cell                          (lat, lon) int64 18MB 2916805 ... 228922
  * lat                           (lat) float64 12kB -74.95 -74.85 ... 74.95
  * lon                           (lon) float64 12kB 285.1 285.1 ... 74.85 74.95
  * core                          (core) int32 28kB 1 2 3 4 ... 6995 6996 6997
  * anvil                         (anvil) int32 40kB 1 2 3 ... 10019 10020 10021
  * core_step                     (core_step) int32 119kB 1 2 3 ... 29852 29853
  * thick_anvil_step              (thick_anvil_step) int32 1MB 1 2 ... 262469
  * thin_anvil_step               (thin_anvil_step) int32 1MB 1 2 ... 310095
Dimensions without coordinates: y, x
Data variables: (12/44)
    core_label                    (t, lat, lon) int32 6GB 0 0 0 0 0 ... 0 0 0 0
    thick_anvil_label             (t, lat, lon) int32 6GB 0 0 0 0 0 ... 0 0 0 0
    thin_anvil_label              (t, lat, lon) int32 6GB 0 0 0 0 0 ... 0 0 0 0
    core_anvil_index              (core) int32 28kB 65 152 117 173 ... 0 0 9564
    anvil_core_count              (anvil) int32 40kB 0 0 7 0 0 0 ... 0 0 0 0 0 0
    core_step_label               (t, y, x) int32 6GB 0 0 0 0 0 0 ... 0 0 0 0 0
    ...                            ...
    thin_anvil_step_pixel_count   (thin_anvil_step) int32 1MB 6096 1704 ... 97
    thin_anvil_step_area          (thin_anvil_step) float32 1MB 3.702e+05 ......
    thin_anvil_start_t            (anvil) datetime64[ns] 80kB 2021-07-01T00:1...
    thin_anvil_end_t              (anvil) datetime64[ns] 80kB 2021-07-02T09:1...
    thin_anvil_lifetime           (anvil) timedelta64[ns] 80kB 1 days 09:00:0...
    thin_anvil_step_t             (thin_anvil_step) datetime64[ns] 2MB 2021-0...

In [35]:
dataset["area"] = xr.DataArray(
    np.tile((6_378 * np.radians(0.1))**2 * np.cos(np.radians(dataset.lat)), [dataset.lon.size, 1]).T, 
    coords={"lat":dataset.lat, "lon":dataset.lon}, 
    dims=("lat", "lon")
)

In [36]:
dataset

<xarray.Dataset> Size: 36GB
Dimensions:                       (t: 659, lat: 1500, lon: 1500, core: 6997,
                                   anvil: 10021, y: 1500, x: 1500,
                                   core_step: 29853, thick_anvil_step: 262469,
                                   thin_anvil_step: 310095)
Coordinates:
    crs                           int64 8B 0
  * t                             (t) datetime64[ns] 5kB 2021-07-01T00:15:00 ...
    cell                          (lat, lon) int64 18MB 2916805 ... 228922
  * lat                           (lat) float64 12kB -74.95 -74.85 ... 74.95
  * lon                           (lon) float64 12kB 285.1 285.1 ... 74.85 74.95
  * core                          (core) int32 28kB 1 2 3 4 ... 6995 6996 6997
  * anvil                         (anvil) int32 40kB 1 2 3 ... 10019 10020 10021
  * core_step                     (core_step) int32 119kB 1 2 3 ... 29852 29853
  * thick_anvil_step              (thick_anvil_step) int32 1MB 1 2 ... 262469
  * thin_anvil_step               (thin_anvil_step) int32 1MB 1 2 ... 310095
Dimensions without coordinates: y, x
Data variables: (12/44)
    core_label                    (t, lat, lon) int32 6GB 0 0 0 0 0 ... 0 0 0 0
    thick_anvil_label             (t, lat, lon) int32 6GB 0 0 0 0 0 ... 0 0 0 0
    thin_anvil_label              (t, lat, lon) int32 6GB 0 0 0 0 0 ... 0 0 0 0
    core_anvil_index              (core) int32 28kB 65 152 117 173 ... 0 0 9564
    anvil_core_count              (anvil) int32 40kB 0 0 7 0 0 0 ... 0 0 0 0 0 0
    core_step_label               (t, y, x) int32 6GB 0 0 0 0 0 0 ... 0 0 0 0 0
    ...                            ...
    thin_anvil_step_pixel_count   (thin_anvil_step) int32 1MB 6096 1704 ... 97
    thin_anvil_step_area          (thin_anvil_step) float32 1MB 3.702e+05 ......
    thin_anvil_start_t            (anvil) datetime64[ns] 80kB 2021-07-01T00:1...
    thin_anvil_end_t              (anvil) datetime64[ns] 80kB 2021-07-02T09:1...
    thin_anvil_lifetime           (anvil) timedelta64[ns] 80kB 1 days 09:00:0...
    thin_anvil_step_t             (thin_anvil_step) datetime64[ns] 2MB 2021-0...

In [37]:
core_total_pixels = np.bincount(dataset.core_label.data.ravel())[dataset.core.data]
add_dataarray_to_ds(
    create_dataarray(
        core_total_pixels,
        ("core",),
        "core_pixel_count",
        long_name="total number of pixels for core",
        dtype=np.int32,
    ),
    dataset,
)


In [38]:
core_step_pixels = np.bincount(dataset.core_step_label.data.ravel())[
    dataset.core_step.data
]
add_dataarray_to_ds(
    create_dataarray(
        core_step_pixels,
        ("core_step",),
        "core_step_pixel_count",
        long_name="total number of pixels for core at time step",
        dtype=np.int32,
    ),
    dataset,
)

In [39]:
core_total_area = labeled_comprehension(
    dataset.area.data[np.newaxis, ...],
    dataset.core_label.data,
    np.nansum,
    index=dataset.core.data,
    dtype=np.float32,
    default=np.nan,
)
add_dataarray_to_ds(
    create_dataarray(
        core_total_area,
        ("core",),
        "core_total_area",
        long_name="total area of core",
        dtype=np.float32,
    ),
    dataset,
)

In [40]:
core_step_area = labeled_comprehension(
    dataset.area.data[np.newaxis, ...],
    dataset.core_step_label.data,
    np.nansum,
    index=dataset.core_step.data,
    dtype=np.float32,
    default=np.nan,
)
add_dataarray_to_ds(
    create_dataarray(
        core_step_area,
        ("core_step",),
        "core_step_area",
        long_name="area of core at time step",
        dtype=np.float32,
    ),
    dataset,
)

In [41]:
core_step_max_area_index = np.asarray(
    [
        dataset.core_step[dataset.core_step_core_index.data == i][
            np.argmax(core_step_area[dataset.core_step_core_index.data == i])
        ]
        for i in dataset.core.data
    ]
)

In [42]:
core_max_area = dataset.core_step_area.loc[core_step_max_area_index].data

add_dataarray_to_ds(
    create_dataarray(
        core_max_area,
        ("core",),
        "core_max_area",
        long_name="maximum area of core",
        dtype=np.float32,
    ),
    dataset,
)

In [43]:
core_start_t = labeled_comprehension(
    dataset.t.data[:, np.newaxis, np.newaxis],
    dataset.core_label.data,
    np.nanmin,
    index=dataset.core.data,
    dtype="datetime64[ns]",
    default=None,
)
add_dataarray_to_ds(
    create_dataarray(
        core_start_t,
        ("core",),
        "core_start_t",
        long_name="initial detection time of core",
        dtype="datetime64[ns]",
    ),
    dataset,
)

core_end_t = labeled_comprehension(
    dataset.t.data[:, np.newaxis, np.newaxis],
    dataset.core_label.data,
    np.nanmax,
    index=dataset.core.data,
    dtype="datetime64[ns]",
    default=None,
)
add_dataarray_to_ds(
    create_dataarray(
        core_end_t,
        ("core",),
        "core_end_t",
        long_name="final detection time of core",
        dtype="datetime64[ns]",
    ),
    dataset,
)

add_dataarray_to_ds(
    create_dataarray(
        core_end_t - core_start_t,
        ("core",),
        "core_lifetime",
        long_name="total lifetime of core",
        dtype="timedelta64[ns]",
    ),
    dataset,
)

In [44]:
core_step_t = labeled_comprehension(
    dataset.t.data[:, np.newaxis, np.newaxis],
    dataset.core_step_label.data,
    np.nanmin,
    index=dataset.core_step.data,
    dtype="datetime64[ns]",
    default=None,
)
add_dataarray_to_ds(
    create_dataarray(
        core_step_t,
        ("core_step",),
        "core_step_t",
        long_name="time of core at time step",
        dtype="datetime64[ns]",
    ),
    dataset,
)

core_max_area_t = dataset.core_step_t.loc[core_step_max_area_index].data
add_dataarray_to_ds(
    create_dataarray(
        core_max_area_t,
        ("core",),
        "core_max_area_t",
        long_name="time of core maximum area",
        dtype="datetime64[ns]",
    ),
    dataset,
)

In [45]:
thick_anvil_step_pixels = np.bincount(dataset.thick_anvil_step_label.data.ravel())[
    dataset.thick_anvil_step.data
]
add_dataarray_to_ds(
    create_dataarray(
        thick_anvil_step_pixels,
        ("thick_anvil_step",),
        "thick_anvil_step_pixel_count",
        long_name="total number of pixels for thick anvil at time step",
        dtype=np.int32,
    ),
    dataset,
)

thick_anvil_total_area = labeled_comprehension(
    dataset.area.data[np.newaxis, ...],
    dataset.thick_anvil_label.data,
    np.nansum,
    index=dataset.anvil.data,
    dtype=np.float32,
    default=np.nan,
)
add_dataarray_to_ds(
    create_dataarray(
        thick_anvil_total_area,
        ("anvil",),
        "thick_anvil_total_area",
        long_name="total area of thick anvil",
        dtype=np.float32,
    ),
    dataset,
)

thick_anvil_step_area = labeled_comprehension(
    dataset.area.data[np.newaxis, ...],
    dataset.thick_anvil_step_label.data,
    np.nansum,
    index=dataset.thick_anvil_step.data,
    dtype=np.float32,
    default=np.nan,
)
add_dataarray_to_ds(
    create_dataarray(
        thick_anvil_step_area,
        ("thick_anvil_step",),
        "thick_anvil_step_area",
        long_name="area of thick anvil at time step",
        dtype=np.float32,
    ),
    dataset,
)

thick_anvil_start_t = labeled_comprehension(
    dataset.t.data[:, np.newaxis, np.newaxis],
    dataset.thick_anvil_label.data,
    np.nanmin,
    index=dataset.anvil.data,
    dtype="datetime64[ns]",
    default=None,
)
add_dataarray_to_ds(
    create_dataarray(
        thick_anvil_start_t,
        ("anvil",),
        "thick_anvil_start_t",
        long_name="initial detection time of thick anvil",
        dtype="datetime64[ns]",
    ),
    dataset,
)

thick_anvil_end_t = labeled_comprehension(
    dataset.t.data[:, np.newaxis, np.newaxis],
    dataset.thick_anvil_label.data,
    np.nanmax,
    index=dataset.anvil.data,
    dtype="datetime64[ns]",
    default=None,
)
add_dataarray_to_ds(
    create_dataarray(
        thick_anvil_end_t,
        ("anvil",),
        "thick_anvil_end_t",
        long_name="final detection time of thick anvil",
        dtype="datetime64[ns]",
    ),
    dataset,
)

add_dataarray_to_ds(
    create_dataarray(
        thick_anvil_end_t - thick_anvil_start_t,
        ("anvil",),
        "thick_anvil_lifetime",
        long_name="total lifetime of thick anvil",
        dtype="timedelta64[ns]",
    ),
    dataset,
)

thick_anvil_step_t = labeled_comprehension(
    dataset.t.data[:, np.newaxis, np.newaxis],
    dataset.thick_anvil_step_label.data,
    np.nanmin,
    index=dataset.thick_anvil_step.data,
    dtype="datetime64[ns]",
    default=None,
)
add_dataarray_to_ds(
    create_dataarray(
        thick_anvil_step_t,
        ("thick_anvil_step",),
        "thick_anvil_step_t",
        long_name="time of thick anvil at time step",
        dtype="datetime64[ns]",
    ),
    dataset,
)

In [46]:
thin_anvil_step_pixels = np.bincount(dataset.thin_anvil_step_label.data.ravel())[
    dataset.thin_anvil_step.data
]
add_dataarray_to_ds(
    create_dataarray(
        thin_anvil_step_pixels,
        ("thin_anvil_step",),
        "thin_anvil_step_pixel_count",
        long_name="total number of pixels for thin anvil at time step",
        dtype=np.int32,
    ),
    dataset,
)

thin_anvil_step_area = labeled_comprehension(
    dataset.area.data[np.newaxis, ...],
    dataset.thin_anvil_step_label.data,
    np.nansum,
    index=dataset.thin_anvil_step.data,
    dtype=np.float32,
    default=np.nan,
)
add_dataarray_to_ds(
    create_dataarray(
        thin_anvil_step_area,
        ("thin_anvil_step",),
        "thin_anvil_step_area",
        long_name="area of thin anvil at time step",
        dtype=np.float32,
    ),
    dataset,
)

thin_anvil_start_t = labeled_comprehension(
    dataset.t.data[:, np.newaxis, np.newaxis],
    dataset.thin_anvil_label.data,
    np.nanmin,
    index=dataset.anvil.data,
    dtype="datetime64[ns]",
    default=None,
)
add_dataarray_to_ds(
    create_dataarray(
        thin_anvil_start_t,
        ("anvil",),
        "thin_anvil_start_t",
        long_name="initial detection time of thin anvil",
        dtype="datetime64[ns]",
    ),
    dataset,
)

thin_anvil_end_t = labeled_comprehension(
    dataset.t.data[:, np.newaxis, np.newaxis],
    dataset.thin_anvil_label.data,
    np.nanmax,
    index=dataset.anvil.data,
    dtype="datetime64[ns]",
    default=None,
)
add_dataarray_to_ds(
    create_dataarray(
        thin_anvil_end_t,
        ("anvil",),
        "thin_anvil_end_t",
        long_name="final detection time of thin anvil",
        dtype="datetime64[ns]",
    ),
    dataset,
)

add_dataarray_to_ds(
    create_dataarray(
        thin_anvil_end_t - thin_anvil_start_t,
        ("anvil",),
        "thin_anvil_lifetime",
        long_name="total lifetime of thin anvil",
        dtype="timedelta64[ns]",
    ),
    dataset,
)

thin_anvil_step_t = labeled_comprehension(
    dataset.t.data[:, np.newaxis, np.newaxis],
    dataset.thin_anvil_step_label.data,
    np.nanmin,
    index=dataset.thin_anvil_step.data,
    dtype="datetime64[ns]",
    default=None,
)
add_dataarray_to_ds(
    create_dataarray(
        thin_anvil_step_t,
        ("thin_anvil_step",),
        "thin_anvil_step_t",
        long_name="time of thin anvil at time step",
        dtype="datetime64[ns]",
    ),
    dataset,
)

In [47]:
dataset

<xarray.Dataset> Size: 36GB
Dimensions:                       (t: 659, lat: 1500, lon: 1500, core: 6997,
                                   anvil: 10021, y: 1500, x: 1500,
                                   core_step: 29853, thick_anvil_step: 262469,
                                   thin_anvil_step: 310095)
Coordinates:
    crs                           int64 8B 0
  * t                             (t) datetime64[ns] 5kB 2021-07-01T00:15:00 ...
    cell                          (lat, lon) int64 18MB 2916805 ... 228922
  * lat                           (lat) float64 12kB -74.95 -74.85 ... 74.95
  * lon                           (lon) float64 12kB 285.1 285.1 ... 74.85 74.95
  * core                          (core) int32 28kB 1 2 3 4 ... 6995 6996 6997
  * anvil                         (anvil) int32 40kB 1 2 3 ... 10019 10020 10021
  * core_step                     (core_step) int32 119kB 1 2 3 ... 29852 29853
  * thick_anvil_step              (thick_anvil_step) int32 1MB 1 2 ... 262469
  * thin_anvil_step               (thin_anvil_step) int32 1MB 1 2 ... 310095
Dimensions without coordinates: y, x
Data variables: (12/44)
    core_label                    (t, lat, lon) int32 6GB 0 0 0 0 0 ... 0 0 0 0
    thick_anvil_label             (t, lat, lon) int32 6GB 0 0 0 0 0 ... 0 0 0 0
    thin_anvil_label              (t, lat, lon) int32 6GB 0 0 0 0 0 ... 0 0 0 0
    core_anvil_index              (core) int32 28kB 65 152 117 173 ... 0 0 9564
    anvil_core_count              (anvil) int32 40kB 0 0 7 0 0 0 ... 0 0 0 0 0 0
    core_step_label               (t, y, x) int32 6GB 0 0 0 0 0 0 ... 0 0 0 0 0
    ...                            ...
    thin_anvil_step_pixel_count   (thin_anvil_step) int32 1MB 6096 1704 ... 97
    thin_anvil_step_area          (thin_anvil_step) float32 1MB 3.702e+05 ......
    thin_anvil_start_t            (anvil) datetime64[ns] 80kB 2021-07-01T00:1...
    thin_anvil_end_t              (anvil) datetime64[ns] 80kB 2021-07-02T09:1...
    thin_anvil_lifetime           (anvil) timedelta64[ns] 80kB 1 days 09:00:0...
    thin_anvil_step_t             (thin_anvil_step) datetime64[ns] 2MB 2021-0...

In [48]:
if (len(dataset.lat.shape)==1) and (len(dataset.lon.shape)==1):
    lons, lats = np.meshgrid(dataset.lon, dataset.lat)
    lat_stack = np.repeat(lats[np.newaxis, ...], dataset.t.size, 0)
    lon_stack = np.repeat(lons[np.newaxis, ...], dataset.t.size, 0)

In [49]:
area_stack = np.repeat(dataset.area.data[np.newaxis, ...], dataset.t.size, 0)
xx, yy = np.meshgrid(dataset.x, dataset.y)
x_stack = np.repeat(xx[np.newaxis, ...], dataset.t.size, 0)
y_stack = np.repeat(yy[np.newaxis, ...], dataset.t.size, 0)


In [ ]:
calculate_label_properties(dataset)

In [61]:
dataset

<xarray.Dataset> Size: 36GB
Dimensions:                                      (t: 659, lat: 1500, lon: 1500,
                                                  core: 6586, anvil: 2488,
                                                  y: 1500, x: 1500,
                                                  core_step: 27988,
                                                  thick_anvil_step: 53870,
                                                  thin_anvil_step: 62519)
Coordinates:
    crs                                          int64 8B 0
  * t                                            (t) datetime64[ns] 5kB 2021-...
    cell                                         (lat, lon) int64 18MB 291680...
  * lat                                          (lat) float64 12kB -74.95 .....
  * lon                                          (lon) float64 12kB 285.1 ......
  * core                                         (core) int32 26kB 1 2 ... 6997
  * anvil                                        (anvil) int32 10kB 65 ... 10015
  * core_step                                    (core_step) int32 112kB 1 .....
  * thick_anvil_step                             (thick_anvil_step) int32 215kB ...
  * thin_anvil_step                              (thin_anvil_step) int32 250kB ...
Dimensions without coordinates: y, x
Data variables: (12/116)
    core_label                                   (t, lat, lon) int32 6GB 0 ... 0
    thick_anvil_label                            (t, lat, lon) int32 6GB 0 ... 0
    thin_anvil_label                             (t, lat, lon) int32 6GB 0 ... 0
    core_anvil_index                             (core) int32 26kB 65 152 ... 0
    anvil_core_count                             (anvil) float64 20kB 1.0 ......
    core_step_label                              (t, y, x) int32 6GB 0 0 ... 0 0
    ...                                           ...
    anvil_initial_core_index                     (anvil) int32 10kB 1 ... 6961
    anvil_no_growth_flag                         (anvil) bool 2kB False ... F...
    anvil_no_initial_core_flag                   (anvil) bool 2kB False ... F...
    core_is_valid                                (core) bool 7kB False ... False
    thick_anvil_is_valid                         (anvil) bool 2kB False ... F...
    thin_anvil_is_valid                          (anvil) bool 2kB False ... F...

In [62]:
from tobac_flow.postprocess import (
    add_validity_flags,
    process_core_properties,
    process_thick_anvil_properties,
    process_thin_anvil_properties,
)
from tobac_flow.utils import (
    remove_orphan_coords,
    filter_cores,
    filter_anvils,
)

In [63]:
dataset = remove_orphan_coords(dataset)
print(datetime.now(), "Removing orphaned items", flush=True)

# Remove invalid cores and process core properties
print(datetime.now(), "Filtering and processing cores", flush=True)
dataset = filter_cores(dataset, verbose=True)
dataset = process_core_properties(dataset)

print(datetime.now(), "Filtering and processing anvils", flush=True)
dataset = filter_anvils(dataset, verbose=True)
dataset = process_thick_anvil_properties(dataset)
dataset = process_thin_anvil_properties(dataset)

print(datetime.now(), "Flagging core and anvil quality", flush=True)
dataset = remove_orphan_coords(dataset)
dataset = add_validity_flags(dataset)


2025-03-10 11:23:03.415880 Removing orphaned items
2025-03-10 11:23:03.416630 Filtering and processing cores
Initial core count: 6586
Valid core cooling: 6586
Valid time gaps: 6586
Valid lifetime: 6586
Valid maximum area: 6586
Valid NaN flagging: 6586
Final core count: 6586
2025-03-10 11:23:31.322662 Filtering and processing anvils
Initial anvil count: 2488
Core present: 2488
Valid NaN flagging: 2488
Valid lifetime: 2488
Valid time gaps: 2488
Valid anvil area: 2488
Valid anvil end time: 2488
Final anvil count: 2488
2025-03-10 11:23:51.165486 Flagging core and anvil quality


In [64]:
print(f"Final core count: {dataset.core.size}")
print(f"Final valid core count: {dataset.core_is_valid.data.sum()}")
print(f"Final anvil count: {dataset.anvil.size}")
print(f"Final valid thick anvil count: {dataset.thick_anvil_is_valid.data.sum()}")
print(f"Final valid thin anvil count: {dataset.thin_anvil_is_valid.data.sum()}")


Final core count: 6586
Final valid core count: 6326
Final anvil count: 2488
Final valid thick anvil count: 1956
Final valid thin anvil count: 2087


In [65]:
dataset

<xarray.Dataset> Size: 36GB
Dimensions:                                      (t: 659, lat: 1500, lon: 1500,
                                                  core: 6586, anvil: 2488,
                                                  y: 1500, x: 1500,
                                                  core_step: 27988,
                                                  thick_anvil_step: 53870,
                                                  thin_anvil_step: 62519)
Coordinates:
    crs                                          int64 8B 0
  * t                                            (t) datetime64[ns] 5kB 2021-...
    cell                                         (lat, lon) int64 18MB 291680...
  * lat                                          (lat) float64 12kB -74.95 .....
  * lon                                          (lon) float64 12kB 285.1 ......
  * core                                         (core) int32 26kB 1 2 ... 6997
  * anvil                                        (anvil) int32 10kB 65 ... 10015
  * core_step                                    (core_step) int32 112kB 1 .....
  * thick_anvil_step                             (thick_anvil_step) int32 215kB ...
  * thin_anvil_step                              (thin_anvil_step) int32 250kB ...
Dimensions without coordinates: y, x
Data variables: (12/116)
    core_label                                   (t, lat, lon) int32 6GB 0 ... 0
    thick_anvil_label                            (t, lat, lon) int32 6GB 0 ... 0
    thin_anvil_label                             (t, lat, lon) int32 6GB 0 ... 0
    core_anvil_index                             (core) int32 26kB 65 152 ... 0
    anvil_core_count                             (anvil) float64 20kB 1.0 ......
    core_step_label                              (t, y, x) int32 6GB 0 0 ... 0 0
    ...                                           ...
    anvil_initial_core_index                     (anvil) int32 10kB 1 ... 6961
    anvil_no_growth_flag                         (anvil) bool 2kB False ... F...
    anvil_no_initial_core_flag                   (anvil) bool 2kB False ... F...
    core_is_valid                                (core) bool 7kB False ... False
    thick_anvil_is_valid                         (anvil) bool 2kB False ... F...
    thin_anvil_is_valid                          (anvil) bool 2kB False ... F...

In [66]:
np.unique(dataset.core_label).size

6998

In [68]:
np.unique(np.where(np.isin(dataset.core_label.data, dataset.core), dataset.core_label.data, 0)).size

6587

In [69]:
dataset.core_label.data = np.where(np.isin(dataset.core_label.data, dataset.core), dataset.core_label.data, 0)
dataset.core_step_label.data = np.where(np.isin(dataset.core_step_label.data, dataset.core_step), dataset.core_step_label.data, 0)

dataset.thick_anvil_label.data = np.where(np.isin(dataset.thick_anvil_label.data, dataset.anvil), dataset.thick_anvil_label.data, 0)
dataset.thick_anvil_step_label.data = np.where(np.isin(dataset.thick_anvil_step_label.data, dataset.thick_anvil_step), dataset.thick_anvil_step_label.data, 0)

dataset.thin_anvil_label.data = np.where(np.isin(dataset.thin_anvil_label.data, dataset.anvil), dataset.thin_anvil_label.data, 0)
dataset.thin_anvil_step_label.data = np.where(np.isin(dataset.thin_anvil_step_label.data, dataset.thin_anvil_step), dataset.thin_anvil_step_label.data, 0)

In [70]:
dataset

<xarray.Dataset> Size: 36GB
Dimensions:                                      (t: 659, lat: 1500, lon: 1500,
                                                  core: 6586, anvil: 2488,
                                                  y: 1500, x: 1500,
                                                  core_step: 27988,
                                                  thick_anvil_step: 53870,
                                                  thin_anvil_step: 62519)
Coordinates:
    crs                                          int64 8B 0
  * t                                            (t) datetime64[ns] 5kB 2021-...
    cell                                         (lat, lon) int64 18MB 291680...
  * lat                                          (lat) float64 12kB -74.95 .....
  * lon                                          (lon) float64 12kB 285.1 ......
  * core                                         (core) int32 26kB 1 2 ... 6997
  * anvil                                        (anvil) int32 10kB 65 ... 10015
  * core_step                                    (core_step) int32 112kB 1 .....
  * thick_anvil_step                             (thick_anvil_step) int32 215kB ...
  * thin_anvil_step                              (thin_anvil_step) int32 250kB ...
Dimensions without coordinates: y, x
Data variables: (12/116)
    core_label                                   (t, lat, lon) int32 6GB 0 ... 0
    thick_anvil_label                            (t, lat, lon) int32 6GB 0 ... 0
    thin_anvil_label                             (t, lat, lon) int32 6GB 0 ... 0
    core_anvil_index                             (core) int32 26kB 65 152 ... 0
    anvil_core_count                             (anvil) float64 20kB 1.0 ......
    core_step_label                              (t, y, x) int32 6GB 0 0 ... 0 0
    ...                                           ...
    anvil_initial_core_index                     (anvil) int32 10kB 1 ... 6961
    anvil_no_growth_flag                         (anvil) bool 2kB False ... F...
    anvil_no_initial_core_flag                   (anvil) bool 2kB False ... F...
    core_is_valid                                (core) bool 7kB False ... False
    thick_anvil_is_valid                         (anvil) bool 2kB False ... F...
    thin_anvil_is_valid                          (anvil) bool 2kB False ... F...

In [71]:
import pathlib

In [72]:
pathlib.Path("../scripts/synsat_tracking_zoom9_2021.nc").stem

'synsat_tracking_zoom9_2021'

In [74]:
"latitude" in dataset

False